# Fall 2025 DS701 Grade Calculation



Final grades will be computed based on the following:<br>

| Percentage | Category |
| ---------- | -------- |
| 15%        | Participation and in-class activities |
| 15%        | Homework assignments |
| 30%        | Midterm challenge |
| 40%        | Final Project (where 24% is for the project and 16% is for the personal contributions)  |


In [ ]:
import pandas as pd

df = pd.read_csv('DS701_Fall_2025_grades.csv')


In [ ]:
print(df.columns)


## Lateness and Penalty Calculations

In [ ]:
import re

def parse_lateness_to_hours(lateness_str):
    """Convert H:M:S lateness string to total hours as float."""
    if pd.isna(lateness_str) or lateness_str == '00:00:00':
        return 0.0
    # Parse H:M:S format
    parts = str(lateness_str).split(':')
    if len(parts) == 3:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = int(parts[2])
        return hours + minutes / 60 + seconds / 3600
    return 0.0

def apply_late_penalty(score, lateness_hours):
    """Apply 10% penalty if lateness is > 0 and < 48 hours."""
    if pd.isna(score):
        return 0.0
    if lateness_hours > 0 and lateness_hours < 48:
        return score * 0.9
    return score


## Find and Assign Categories to Assignments

In [ ]:
# Get all unique assignment names by finding columns that don't have suffixes
assignment_columns = []
for col in df.columns:
    if ' - Max Points' in col:
        assignment_name = col.replace(' - Max Points', '')
        assignment_columns.append(assignment_name)

print("Found assignments:")
for a in assignment_columns:
    print(f"  - {a}")


In [ ]:
# Define assignment categories
midterm_challenge_assignments = [
    'Midterm Challenge -- Sports',
    'Midterm Challenge -- Health'
]

final_project_assignments = [
    'Project Final Submission, Repo, Presentation and Report',
    'Final Project Personal Contributions'
]

# In-class activities = specific assignments plus any with 'in-class activity' in name
in_class_specific = [
    'AI Use Reflection',
    'Intro Survey',
    'Spark Project Pitches and Preference Form',
    'Spark Midterm Survey'
]

in_class_activities = [
    a for a in assignment_columns 
    if 'in-class activity' in a.lower() or a in in_class_specific
]

# Homework = everything except midterm challenge, final project, and in-class activities
homework_assignments = [
    a for a in assignment_columns 
    if a not in midterm_challenge_assignments 
    and a not in final_project_assignments
    and a not in in_class_activities
]

print("Homework Assignments:")
for a in homework_assignments:
    print(f"  - {a}")
    
print("\nIn-Class Activities:")
for a in in_class_activities:
    print(f"  - {a}")
    
print("\nMidterm Challenge Assignments:")
for a in midterm_challenge_assignments:
    print(f"  - {a}")
    
print("\nFinal Project Assignments:")
for a in final_project_assignments:
    print(f"  - {a}")


## Score Calculations

In [ ]:
# Calculate scores with late penalties applied
def get_adjusted_score(row, assignment_name):
    """Get the score for an assignment with late penalty applied if applicable."""
    score_col = assignment_name
    lateness_col = f"{assignment_name} - Lateness (H:M:S)"
    
    if score_col not in df.columns:
        return 0.0
    
    score = row[score_col]
    if pd.isna(score):
        return 0.0
    
    # Ignore lateness for these assignment types
    if 'Individual Reflection' in assignment_name:
        return score
    if 'Personal Contributions' in assignment_name:
        return score
    if 'Project Final Submission' in assignment_name:
        return score
    
    # Get lateness if available
    if lateness_col in df.columns:
        lateness_hours = parse_lateness_to_hours(row[lateness_col])
        return apply_late_penalty(score, lateness_hours)
    else:
        return score

# Calculate Total Homework Score (with late penalties)
def calculate_homework_total(row):
    total = 0.0
    for assignment in homework_assignments:
        total += get_adjusted_score(row, assignment)
    return total

df['Total Homework Score'] = df.apply(calculate_homework_total, axis=1)

# Calculate In-Class Activities Total (with penalties)
def calculate_in_class_total(row):
    total = 0.0
    for assignment in in_class_activities:
        total += get_adjusted_score(row, assignment)
    return total

df['In-Class Activities Score'] = df.apply(calculate_in_class_total, axis=1)

# Calculate Final Project Total (sum of Final Project + Personal Contributions, with penalties)
def calculate_final_project_total(row):
    final_project_score = get_adjusted_score(row, 'Project Final Submission, Repo, Presentation and Report')
    personal_contributions_score = get_adjusted_score(row, 'Final Project Personal Contributions')
    return final_project_score + personal_contributions_score

df['Final Project Total Score'] = df.apply(calculate_final_project_total, axis=1)

# Calculate max possible points for each category
max_homework_points = 0
for assignment in homework_assignments:
    max_col = f"{assignment} - Max Points"
    if max_col in df.columns:
        max_val = df[max_col].max()
        if pd.notna(max_val):
            max_homework_points += max_val

max_in_class_points = 0
for assignment in in_class_activities:
    max_col = f"{assignment} - Max Points"
    if max_col in df.columns:
        max_val = df[max_col].max()
        if pd.notna(max_val):
            max_in_class_points += max_val

max_sports_points = df['Midterm Challenge -- Sports - Max Points'].max() if 'Midterm Challenge -- Sports - Max Points' in df.columns else 0
max_health_points = df['Midterm Challenge -- Health - Max Points'].max() if 'Midterm Challenge -- Health - Max Points' in df.columns else 0

max_final_project_points = 0
if 'Project Final Submission, Repo, Presentation and Report - Max Points' in df.columns:
    max_final_project_points += df['Project Final Submission, Repo, Presentation and Report - Max Points'].max()
if 'Final Project Personal Contributions - Max Points' in df.columns:
    max_final_project_points += df['Final Project Personal Contributions - Max Points'].max()

# Calculate percentages for each category
df['Total Homework Percentage'] = (df['Total Homework Score'] / max_homework_points * 100) if max_homework_points > 0 else 0
df['In-Class Activities Percentage'] = (df['In-Class Activities Score'] / max_in_class_points * 100) if max_in_class_points > 0 else 0
df['Final Project Percentage'] = (df['Final Project Total Score'] / max_final_project_points * 100) if max_final_project_points > 0 else 0

# Calculate Personal Contributions score and percentage (out of 40 points)
def get_personal_contributions_score(row):
    return get_adjusted_score(row, 'Final Project Personal Contributions')

df['Final Project Personal Contributions Score'] = df.apply(get_personal_contributions_score, axis=1)
max_personal_contributions = 40.0
df['Final Project Personal Contributions Percentage'] = (df['Final Project Personal Contributions Score'] / max_personal_contributions * 100)

# Calculate Midterm Challenge Score (highest percentage of Sports or Health, with penalties)
def calculate_midterm_challenge(row):
    sports_score = get_adjusted_score(row, 'Midterm Challenge -- Sports')
    health_score = get_adjusted_score(row, 'Midterm Challenge -- Health')
    
    # Calculate percentages
    sports_pct = (sports_score / max_sports_points * 100) if max_sports_points > 0 and sports_score > 0 else 0
    health_pct = (health_score / max_health_points * 100) if max_health_points > 0 and health_score > 0 else 0
    
    # Return the score with the highest percentage
    if sports_pct >= health_pct:
        return sports_score, sports_pct, max_sports_points
    else:
        return health_score, health_pct, max_health_points

# Apply the function and unpack results
midterm_results = df.apply(calculate_midterm_challenge, axis=1, result_type='expand')
df['Midterm Challenge Score'] = midterm_results[0]
df['Midterm Challenge Percentage'] = midterm_results[1]
df['Midterm Challenge Max Points'] = midterm_results[2]

# Calculate Team portion of Final Project (separate from Personal Contributions)
def get_team_project_score(row):
    return get_adjusted_score(row, 'Project Final Submission, Repo, Presentation and Report')

df['Final Project Team Score'] = df.apply(get_team_project_score, axis=1)
max_team_project = df['Project Final Submission, Repo, Presentation and Report - Max Points'].max() if 'Project Final Submission, Repo, Presentation and Report - Max Points' in df.columns else 60.0
df['Final Project Team Percentage'] = (df['Final Project Team Score'] / max_team_project * 100)

print(f"Max Team Project points: {max_team_project}")
print(f"Max Personal Contributions points: {max_personal_contributions}")

# Calculate Overall Course Grade with weights:
# Homework: 15%, Midterm: 30%, In-Class: 15%, Team Project: 24%, Personal Contributions: 16%
def calculate_course_grade(row):
    homework_weighted = row['Total Homework Percentage'] * 0.15
    midterm_weighted = row['Midterm Challenge Percentage'] * 0.30
    in_class_weighted = row['In-Class Activities Percentage'] * 0.15
    team_project_weighted = row['Final Project Team Percentage'] * 0.24
    personal_contrib_weighted = row['Final Project Personal Contributions Percentage'] * 0.16
    
    overall = homework_weighted + midterm_weighted + in_class_weighted + team_project_weighted + personal_contrib_weighted
    return overall

df['Overall Course Grade Percentage'] = df.apply(calculate_course_grade, axis=1)

print("\n=== NEW COLUMNS ADDED ===")
print("  - Total Homework Score & Percentage")
print("  - In-Class Activities Score & Percentage")
print("  - Midterm Challenge Score & Percentage (selected by highest percentage)")
print("  - Midterm Challenge Max Points (for selected option)")
print("  - Final Project Total Score & Percentage")
print("  - Final Project Team Score & Percentage")
print("  - Final Project Personal Contributions Score & Percentage (out of 40)")
print("  - Overall Course Grade Percentage")
print("\n=== COURSE GRADE WEIGHTS ===")
print("Homework: 15%")
print("Midterm Challenge: 30%")
print("In-Class Activities: 15%")
print("Final Project Team: 24%")
print("Final Project Personal Contributions: 16%")
print("Total: 100%")


In [ ]:
# Debug: Check max points calculation for homework
print("=== HOMEWORK MAX POINTS BREAKDOWN ===\n")
print(f"Total max homework points used: {max_homework_points}\n")

print("Individual homework assignments and their max points:")
homework_max_breakdown = []
for assignment in homework_assignments:
    max_col = f"{assignment} - Max Points"
    if max_col in df.columns:
        max_val = df[max_col].max()
        if pd.notna(max_val):
            homework_max_breakdown.append((assignment, max_val))
            print(f"  {assignment}: {max_val}")
        else:
            print(f"  {assignment}: NO MAX POINTS DATA")
    else:
        print(f"  {assignment}: MAX POINTS COLUMN NOT FOUND")

print(f"\nSum of all homework max points: {sum([x[1] for x in homework_max_breakdown])}")

# Check for students with >100% homework
over_100 = df[df['Total Homework Percentage'] > 100][['Name', 'Total Homework Score', 'Total Homework Percentage']].head(5)
if len(over_100) > 0:
    print(f"\n=== STUDENTS WITH >100% HOMEWORK ===")
    print(over_100)
else:
    print("\n=== No students with >100% homework ===")

# Show a sample student's homework breakdown
print("\n=== SAMPLE STUDENT HOMEWORK BREAKDOWN ===")
sample_student = df.iloc[0]
print(f"Student: {sample_student['Name']}")
print(f"Total Homework Score: {sample_student['Total Homework Score']}")
print(f"Total Homework Percentage: {sample_student['Total Homework Percentage']:.2f}%")
print("\nIndividual homework assignments:")
for assignment in homework_assignments[:5]:  # Show first 5
    score = sample_student[assignment] if assignment in df.columns else 'N/A'
    max_col = f"{assignment} - Max Points"
    max_pts = sample_student[max_col] if max_col in df.columns else 'N/A'
    print(f"  {assignment}: {score}/{max_pts}")


In [ ]:
# Check which assignments have scores exceeding max points (extra credit)
print("=== CHECKING FOR EXTRA CREDIT ===\n")

for assignment in homework_assignments:
    max_col = f"{assignment} - Max Points"
    if assignment in df.columns and max_col in df.columns:
        # Find students who scored above max
        max_points = df[max_col].max()
        if pd.notna(max_points):
            above_max = df[df[assignment] > max_points]
            if len(above_max) > 0:
                print(f"{assignment}:")
                print(f"  Max points: {max_points}")
                print(f"  Students scoring above max: {len(above_max)}")
                print(f"  Highest score: {df[assignment].max()}")
                print()

# Show Aadi Sharma's full homework breakdown
print("\n=== AADI SHARMA'S COMPLETE HOMEWORK BREAKDOWN ===")
aadi = df[df['Name'] == 'Aadi Sharma'].iloc[0]
total_calculated = 0
for assignment in homework_assignments:
    if assignment in df.columns:
        score = aadi[assignment] if pd.notna(aadi[assignment]) else 0
        max_col = f"{assignment} - Max Points"
        max_pts = aadi[max_col] if max_col in df.columns and pd.notna(aadi[max_col]) else 0
        total_calculated += score
        extra = " (EXTRA CREDIT!)" if score > max_pts else ""
        print(f"  {assignment}: {score}/{max_pts}{extra}")

print(f"\nTotal calculated: {total_calculated}")
print(f"Total in dataframe: {aadi['Total Homework Score']}")
print(f"Max possible (no extra credit): {max_homework_points}")


In [ ]:
# Display the new columns with student info
result_df = df[['Name', 'SID', 'Email', 
                'Overall Course Grade Percentage',
                'Total Homework Score', 'Total Homework Percentage',
                'In-Class Activities Score', 'In-Class Activities Percentage', 
                'Midterm Challenge Score', 'Midterm Challenge Percentage', 'Midterm Challenge Max Points',
                'Final Project Team Score', 'Final Project Team Percentage',
                'Final Project Personal Contributions Score', 'Final Project Personal Contributions Percentage']]

print(f"Total students: {len(result_df)}")
print(f"\nHomework assignments count: {len(homework_assignments)}")
print(f"In-class activities count: {len(in_class_activities)}")
            
print(f"\nMax possible homework points: {max_homework_points}")
print(f"Max possible in-class activities points: {max_in_class_points}")
print(f"Max Midterm Challenge points: Sports={max_sports_points}, Health={max_health_points}")
print(f"Max Team Project points: {max_team_project}")
print(f"Max Personal Contributions points: {max_personal_contributions}")

result_df.head(20)


In [ ]:
# Show sample course grade calculations
print("=== SAMPLE COURSE GRADE CALCULATIONS ===\n")

# Show first 3 students
for i in range(min(3, len(df))):
    student = df.iloc[i]
    print(f"Student: {student['Name']}")
    print(f"  Homework: {student['Total Homework Percentage']:.2f}% × 0.15 = {student['Total Homework Percentage'] * 0.15:.2f}")
    print(f"  Midterm: {student['Midterm Challenge Percentage']:.2f}% × 0.30 = {student['Midterm Challenge Percentage'] * 0.30:.2f}")
    print(f"  In-Class: {student['In-Class Activities Percentage']:.2f}% × 0.15 = {student['In-Class Activities Percentage'] * 0.15:.2f}")
    print(f"  Team Project: {student['Final Project Team Percentage']:.2f}% × 0.24 = {student['Final Project Team Percentage'] * 0.24:.2f}")
    print(f"  Personal Contrib: {student['Final Project Personal Contributions Percentage']:.2f}% × 0.16 = {student['Final Project Personal Contributions Percentage'] * 0.16:.2f}")
    print(f"  Overall Course Grade: {student['Overall Course Grade Percentage']:.2f}%")
    print()

print(f"\n=== OVERALL COURSE GRADE STATISTICS ===")
print(f"Mean: {df['Overall Course Grade Percentage'].mean():.2f}%")
print(f"Median: {df['Overall Course Grade Percentage'].median():.2f}%")
print(f"Min: {df['Overall Course Grade Percentage'].min():.2f}%")
print(f"Max: {df['Overall Course Grade Percentage'].max():.2f}%")
print(f"Std Dev: {df['Overall Course Grade Percentage'].std():.2f}%")


In [ ]:
# Show examples of late penalty applications
# Find students who had late submissions within the 48-hour penalty window

print("Examples of late penalty applications (lateness > 0 and < 48 hours):")
print("(Note: 'Individual Reflection', 'Personal Contributions', and 'Project Final Submission' assignments are excluded from late penalties)\n")

# Check a few specific assignments for late penalties
for assignment in ['Linear Algebra', 'K-Means Clustering', 'Hierarchical Clustering and GMMs']:
    score_col = assignment
    lateness_col = f"{assignment} - Lateness (H:M:S)"
    
    if lateness_col in df.columns:
        # Find students with late submissions in penalty range
        for idx, row in df.iterrows():
            lateness_hours = parse_lateness_to_hours(row[lateness_col])
            if lateness_hours > 0 and lateness_hours < 48:
                original_score = row[score_col]
                adjusted_score = apply_late_penalty(original_score, lateness_hours)
                print(f"{row['Name']} - {assignment}:")
                print(f"  Original score: {original_score}, Lateness: {row[lateness_col]} ({lateness_hours:.2f}h)")
                print(f"  Adjusted score: {adjusted_score} (10% penalty applied)")
                print()
                break  # Just show one example per assignment


In [ ]:
# Summary statistics for the new score columns
print("Summary Statistics for Overall Course Grade:\n")
print(df[['Overall Course Grade Percentage']].describe())

print("\n\nSummary Statistics for Scores:\n")
score_cols = ['Total Homework Score', 'In-Class Activities Score', 'Midterm Challenge Score', 'Final Project Team Score', 'Final Project Personal Contributions Score']
print(df[score_cols].describe())

print("\n\nSummary Statistics for Percentages:\n")
pct_cols = ['Total Homework Percentage', 'In-Class Activities Percentage', 'Midterm Challenge Percentage', 'Final Project Team Percentage', 'Final Project Personal Contributions Percentage']
print(df[pct_cols].describe())


In [ ]:
# Find students with late submissions past 48 hours
# (excluding certain assignment types that don't have late penalties)
print("Students with late submissions past 48 hours:\n")
print("(Excluding 'Individual Reflection', 'Personal Contributions', and 'Project Final Submission' assignments)\n")

late_records = []

for assignment in assignment_columns:
    # Skip assignments that don't have late penalties
    if 'Individual Reflection' in assignment:
        continue
    if 'Personal Contributions' in assignment:
        continue
    if 'Personal Reflections' in assignment:
        continue
    if 'Project Final Submission' in assignment:
        continue
    
    lateness_col = f"{assignment} - Lateness (H:M:S)"
    score_col = assignment
    
    if lateness_col in df.columns:
        for idx, row in df.iterrows():
            lateness_hours = parse_lateness_to_hours(row[lateness_col])
            if lateness_hours >= 48:
                late_records.append({
                    'Name': row['Name'],
                    'Email': row['Email'],
                    'Assignment': assignment,
                    'Score': row[score_col],
                    'Max Points': row[f"{assignment} - Max Points"] if f"{assignment} - Max Points" in df.columns else None,
                    'Lateness': row[lateness_col],
                    'Lateness (hours)': round(lateness_hours, 2)
                })

# Create a dataframe of late submissions
late_df = pd.DataFrame(late_records)

if len(late_df) > 0:
    # Sort by student name then assignment
    late_df = late_df.sort_values(['Name', 'Assignment'])
    print(f"Total late submissions (>= 48 hours): {len(late_df)}")
    print(f"Number of unique students affected: {late_df['Name'].nunique()}")
    print()
    
    # Display all late submissions
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', 50)
    display(late_df)
else:
    print("No late submissions past 48 hours found.")


In [ ]:
# Write the entire dataframe to an Excel file
output_file = 'DS701_Fall_2025_grades_processed.xlsx'

df.to_excel(output_file, index=False, engine='openpyxl')

print(f"Dataframe written to: {output_file}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")



In [ ]:
# Write summary scores to a separate Excel file
summary_columns = [
    'Name', 'SID', 'Email', 'Sections',
    'Overall Course Grade Percentage',
    'Total Homework Score', 'Total Homework Percentage',
    'In-Class Activities Score', 'In-Class Activities Percentage',
    'Midterm Challenge Score', 'Midterm Challenge Percentage', 'Midterm Challenge Max Points',
    'Final Project Team Score', 'Final Project Team Percentage',
    'Final Project Personal Contributions Score', 'Final Project Personal Contributions Percentage'
]

# Select only columns that exist in the dataframe
available_summary_cols = [col for col in summary_columns if col in df.columns]

summary_df = df[available_summary_cols]
summary_output_file = 'summary_scores.xlsx'

summary_df.to_excel(summary_output_file, index=False, engine='openpyxl')

print(f"\nSummary scores written to: {summary_output_file}")
print(f"Columns included: {len(available_summary_cols)}")
print(f"  - {', '.join(available_summary_cols[:4])} (identifiers)")
print(f"  - Overall Course Grade Percentage (weighted)")
print(f"  - {len(available_summary_cols) - 5} category score/percentage columns")
print(f"\nCourse Grade Breakdown:")
print(f"  - Homework: 15%")
print(f"  - Midterm Challenge: 30%")
print(f"  - In-Class Activities: 15%")
print(f"  - Final Project Team: 24%")
print(f"  - Final Project Personal Contributions: 16%")
